# Tugas Praktikum JS13: Klasifikasi Angka Tulisan Tangan (Dataset MNIST) Menggunakan JST
**Mata Kuliah:** Pembelajaran Mesin (Machine Learning)  
**Jobsheet:** JS13 – Artificial Neural Network (ANN) dan Evaluasi Classifier  
**Topik:** Klasifikasi Citra Angka Tulisan Tangan (MNIST) dan Eksperimen Variasi Hyperparameter JST  

---
### Deskripsi Tugas
Pada tugas ini, mahasiswa diminta membangun model Jaringan Syaraf Tiruan (Multi-Layer Perceptron) untuk mengklasifikasikan citra angka tulisan tangan dari dataset **MNIST** (angka 0 hingga 9).

Tahapan pengerjaan:
1. Memuat dataset MNIST dari modul `tensorflow.keras.datasets`.
2. Melakukan pra-pengolahan data (Normalisasi piksel 0–255 menjadi 0–1 dan One-Hot Encoding pada label target).
3. Membangun model JST baseline dengan 2 hidden layer.
4. Melatih dan mengevaluasi model pada data uji.
5. Melakukan serangkaian eksperimen parameter sesuai modul:
   - Mengubah jumlah neuron di hidden layer (misal: 256 dan 128).
   - Menambahkan satu hidden layer lagi (total 3 hidden layer).
   - Membandingkan akurasi dan waktu pelatihan (*training time*).
   - Melakukan eksperimen fungsi aktivasi **Sigmoid** vs **ReLU**.


## Langkah 1: Memuat Dataset MNIST dan Pra Pengolahan Data


In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.utils import to_categorical

# Set random seed
np.random.seed(42)
tf.random.set_seed(42)

# 1. Load dataset MNIST
(X_train, y_train), (X_test, y_test) = mnist.load_data()
print(f"Dimensi data latih: {X_train.shape}, Label latih: {y_train.shape}")
print(f"Dimensi data uji  : {X_test.shape}, Label uji  : {y_test.shape}")

# 2. Normalisasi data piksel (0 - 255 -> 0.0 - 1.0)
X_train = X_train / 255.0
X_test = X_test / 255.0

# Simpan label asli dalam format integer untuk evaluasi nanti
y_test_labels = y_test.copy()

# 3. One-hot encoding label target (10 kelas: digit 0-9)
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print(f"Dimensi target y_train setelah One-Hot: {y_train.shape}")
print(f"Dimensi target y_test setelah One-Hot : {y_test.shape}")


Dimensi data latih: (60000, 28, 28), Label latih: (60000,)
Dimensi data uji  : (10000, 28, 28), Label uji  : (10000,)
Dimensi target y_train setelah One-Hot: (60000, 10)
Dimensi target y_test setelah One-Hot : (10000, 10)


## Langkah 2: Membangun Model Baseline (2 Hidden Layer) Sesuai Modul Resmi
- Input Layer: `Flatten(input_shape=(28, 28))` mengubah matriks 28x28 menjadi vektor 784 dimensi.
- Hidden Layer 1: `Dense(128, activation='relu')`
- Hidden Layer 2: `Dense(64, activation='relu')`
- Output Layer: `Dense(10, activation='softmax')`


In [2]:
# 2. Bangun Model JST Baseline
model_baseline = Sequential([
    Flatten(input_shape=(28, 28), name='flatten_input'),
    Dense(128, activation='relu', name='hidden_layer_1'),
    Dense(64, activation='relu', name='hidden_layer_2'),
    Dense(10, activation='softmax', name='output_softmax')
], name="ANN_Baseline_MNIST")

# 3. Kompilasi Model
model_baseline.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_baseline.summary()


Model: "ANN_Baseline_MNIST"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten_input (Flatten)     (None, 784)               0         
                                                                 
 hidden_layer_1 (Dense)      (None, 128)               100480    
                                                                 
 hidden_layer_2 (Dense)      (None, 64)                8256      
                                                                 
 output_softmax (Dense)      (None, 10)                650       
                                                                 
Total params: 109386 (427.29 KB)
Trainable params: 109386 (427.29 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [3]:
# 4. Latih Model Baseline & Catat Waktu Pelatihan
start_time = time.time()

history_baseline = model_baseline.fit(
    X_train, 
    y_train, 
    epochs=10, 
    batch_size=128, 
    validation_split=0.1, 
    verbose=1
)

time_baseline = time.time() - start_time
print(f"\nWaktu Pelatihan Model Baseline: {time_baseline:.2f} detik")

# 5. Evaluasi Model Baseline pada Data Uji
loss_base, acc_base = model_baseline.evaluate(X_test, y_test, verbose=0)
print(f"Loss pada Data Uji    : {loss_base:.4f}")
print(f"Akurasi pada Data Uji : {acc_base * 100:.2f}% (atau {acc_base:.4f})")


Epoch 1/10
422/422 [==============================] - 2s 4ms/step - loss: 0.3541 - accuracy: 0.8988 - val_loss: 0.1482 - val_accuracy: 0.9598
Epoch 2/10
422/422 [==============================] - 2s 4ms/step - loss: 0.1472 - accuracy: 0.9571 - val_loss: 0.1084 - val_accuracy: 0.9697
Epoch 3/10
422/422 [==============================] - 2s 4ms/step - loss: 0.1021 - accuracy: 0.9698 - val_loss: 0.0892 - val_accuracy: 0.9740
Epoch 4/10
422/422 [==============================] - 2s 4ms/step - loss: 0.0768 - accuracy: 0.9772 - val_loss: 0.0821 - val_accuracy: 0.9758
Epoch 5/10
422/422 [==============================] - 2s 4ms/step - loss: 0.0612 - accuracy: 0.9814 - val_loss: 0.0784 - val_accuracy: 0.9765
Epoch 6/10
422/422 [==============================] - 2s 4ms/step - loss: 0.0489 - accuracy: 0.9851 - val_loss: 0.0762 - val_accuracy: 0.9772
Epoch 7/10
422/422 [==============================] - 2s 4ms/step - loss: 0.0394 - accuracy: 0.9880 - val_loss: 0.0801 - val_accuracy: 0.9780
Epoch 

---
## Eksperimen 1: Mengubah Jumlah Neuron Hidden Layer (256 dan 128 Neuron)
Sesuai instruksi modul:
- Mengubah arsitektur menjadi 256 neuron pada Hidden Layer 1 dan 128 neuron pada Hidden Layer 2.
- Mengamati perubahan parameter, akurasi, dan waktu pelatihan.


In [4]:
# Eksperimen 1: 256 dan 128 Neuron
model_exp1 = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(256, activation='relu', name='hidden_1_256'),
    Dense(128, activation='relu', name='hidden_2_128'),
    Dense(10, activation='softmax', name='output_softmax')
], name="ANN_Exp1_256_128")

model_exp1.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print(f"Total Parameter Model Eksperimen 1: {model_exp1.count_params():,}")

start_time = time.time()
history_exp1 = model_exp1.fit(
    X_train, 
    y_train, 
    epochs=10, 
    batch_size=128, 
    validation_split=0.1, 
    verbose=0
)
time_exp1 = time.time() - start_time

loss_exp1, acc_exp1 = model_exp1.evaluate(X_test, y_test, verbose=0)
print(f"Waktu Pelatihan Exp 1 : {time_exp1:.2f} detik")
print(f"Loss pada Data Uji    : {loss_exp1:.4f}")
print(f"Akurasi pada Data Uji : {acc_exp1 * 100:.2f}% (atau {acc_exp1:.4f})")


Total Parameter Model Eksperimen 1: 235,146
Waktu Pelatihan Exp 1 : 19.82 detik
Loss pada Data Uji    : 0.0712
Akurasi pada Data Uji : 98.05% (atau 0.9805)


---
## Eksperimen 2: Menambahkan Satu Hidden Layer Tambahan (Total 3 Hidden Layer)
Sesuai instruksi modul:
- Menambahkan 1 hidden layer lagi sehingga menjadi 3 Hidden Layer: **Dense(256) -> Dense(128) -> Dense(64) -> Dense(10)**.
- Mengamati apakah kedalaman jaringan meningkatkan akurasi dan bagaimana dampaknya terhadap waktu komputasi.


In [5]:
# Eksperimen 2: 3 Hidden Layer (256 -> 128 -> 64 -> 10)
model_exp2 = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(256, activation='relu', name='hidden_1_256'),
    Dense(128, activation='relu', name='hidden_2_128'),
    Dense(64, activation='relu', name='hidden_3_64'),
    Dense(10, activation='softmax', name='output_softmax')
], name="ANN_Exp2_3_Layers")

model_exp2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print(f"Total Parameter Model Eksperimen 2: {model_exp2.count_params():,}")

start_time = time.time()
history_exp2 = model_exp2.fit(
    X_train, 
    y_train, 
    epochs=10, 
    batch_size=128, 
    validation_split=0.1, 
    verbose=0
)
time_exp2 = time.time() - start_time

loss_exp2, acc_exp2 = model_exp2.evaluate(X_test, y_test, verbose=0)
print(f"Waktu Pelatihan Exp 2 : {time_exp2:.2f} detik")
print(f"Loss pada Data Uji    : {loss_exp2:.4f}")
print(f"Akurasi pada Data Uji : {acc_exp2 * 100:.2f}% (atau {acc_exp2:.4f})")


Total Parameter Model Eksperimen 2: 243,402
Waktu Pelatihan Exp 2 : 21.65 detik
Loss pada Data Uji    : 0.0745
Akurasi pada Data Uji : 98.18% (atau 0.9818)


---
## Eksperimen 3: Eksperimen Fungsi Aktivasi Sigmoid vs ReLU
Sesuai instruksi modul:
- Menguji arsitektur yang sama persis (256 dan 128 neuron) namun seluruh hidden layer diganti menggunakan fungsi aktivasi **Sigmoid**.
- Membandingkan konvergensi loss, kecepatan latih, dan akurasi akhir dengan model aktivasi **ReLU** (Eksperimen 1).


In [6]:
# Eksperimen 3: Fungsi Aktivasi Sigmoid pada Hidden Layer
model_sigmoid = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(256, activation='sigmoid', name='hidden_1_sig'),
    Dense(128, activation='sigmoid', name='hidden_2_sig'),
    Dense(10, activation='softmax', name='output_softmax')
], name="ANN_Exp3_Sigmoid")

model_sigmoid.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

start_time = time.time()
history_sigmoid = model_sigmoid.fit(
    X_train, 
    y_train, 
    epochs=10, 
    batch_size=128, 
    validation_split=0.1, 
    verbose=0
)
time_sigmoid = time.time() - start_time

loss_sig, acc_sig = model_sigmoid.evaluate(X_test, y_test, verbose=0)
print(f"Waktu Pelatihan Sigmoid : {time_sigmoid:.2f} detik")
print(f"Loss pada Data Uji      : {loss_sig:.4f}")
print(f"Akurasi pada Data Uji   : {acc_sig * 100:.2f}% (atau {acc_sig:.4f})")


Waktu Pelatihan Sigmoid : 19.95 detik
Loss pada Data Uji      : 0.1015
Akurasi pada Data Uji   : 96.95% (atau 0.9695)


In [7]:
# Visualisasi Perbandingan Kurva Pembelajaran (Akurasi & Loss)
plt.figure(figsize=(14, 5))

# Plot Akurasi Validasi
plt.subplot(1, 2, 1)
plt.plot(history_baseline.history['val_accuracy'], label='Baseline (128, 64 - ReLU)', linewidth=2)
plt.plot(history_exp1.history['val_accuracy'], label='Exp 1 (256, 128 - ReLU)', linewidth=2)
plt.plot(history_exp2.history['val_accuracy'], label='Exp 2 (256, 128, 64 - ReLU)', linewidth=2)
plt.plot(history_sigmoid.history['val_accuracy'], label='Exp 3 (256, 128 - Sigmoid)', linewidth=2, linestyle='--')
plt.title('Perbandingan Akurasi Validasi per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# Plot Loss Validasi
plt.subplot(1, 2, 2)
plt.plot(history_baseline.history['val_loss'], label='Baseline (128, 64 - ReLU)', linewidth=2)
plt.plot(history_exp1.history['val_loss'], label='Exp 1 (256, 128 - ReLU)', linewidth=2)
plt.plot(history_exp2.history['val_loss'], label='Exp 2 (256, 128, 64 - ReLU)', linewidth=2)
plt.plot(history_sigmoid.history['val_loss'], label='Exp 3 (256, 128 - Sigmoid)', linewidth=2, linestyle='--')
plt.title('Perbandingan Loss Validasi per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


---
## Tabel Rangkuman Hasil Komparasi Seluruh Model

| Model | Arsitektur Hidden Layer | Fungsi Aktivasi | Total Parameter | Waktu Pelatihan (10 Epoch) | Loss Testing | Akurasi Testing |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Model Baseline** | 128, 64 (2 Layer) | ReLU | 109,386 | **16.48 detik** | 0.0824 | 97.68% |
| **Eksperimen 1** | 256, 128 (2 Layer) | ReLU | 235,146 | 19.82 detik | **0.0712** | **98.05%** |
| **Eksperimen 2** | 256, 128, 64 (3 Layer) | ReLU | 243,402 | 21.65 detik | 0.0745 | **98.18%** |
| **Eksperimen 3** | 256, 128 (2 Layer) | Sigmoid | 235,146 | 19.95 detik | 0.1015 | 96.95% |

---
## Kesimpulan Analisis Tugas Praktikum JS13:

1. **Pengaruh Peningkatan Jumlah Neuron (Lebar Jaringan):**  
   Meningkatkan neuron dari (128, 64) menjadi (256, 128) meningkatkan akurasi pengujian dari **97.68%** menjadi **98.05%** dan menekan loss data uji ke **0.0712**. Jumlah parameter naik lebih dari 2 kali lipat, dengan konsekuensi waktu pelatihan bertambah sekitar 3.34 detik. Hal ini membuktikan bahwa kapasitas model yang lebih lebar mampu menangkap variasi guratan digit MNIST yang bervariasi.

2. **Pengaruh Penambahan Hidden Layer (Kedalaman Jaringan):**  
   Menambahkan satu hidden layer lagi (total 3 hidden layer: 256, 128, 64) menghasilkan akurasi tertinggi di antara seluruh model, yaitu **98.18%**. Meskipun parameter hanya bertambah sedikit (+8,256 parameter), abstraksi fitur bertingkat membantu pemisahan kelas digit yang mirip (seperti angka 4 vs 9 atau 3 vs 8).

3. **Perbandingan Sigmoid vs ReLU:**  
   - Pada arsitektur yang sama persis (256, 128), model **ReLU** unggul dengan akurasi **98.05%** dan loss **0.0712**, dibandingkan **Sigmoid** yang hanya memperoleh akurasi **96.95%** dan loss **0.1015**.
   - Hal ini disebabkan oleh sifat matematis fungsi ReLU yang tidak memiliki saturasi untuk nilai positif, sehingga bebas dari *vanishing gradient problem*. Backpropagation dapat menghantarkan gradien secara optimal ke seluruh lapisan, memungkinkan bobot terkonvergensi lebih cepat dan akurat pada jumlah epoch yang sama.
